# naja-schematic in Google Colab — test notebook

Checks the `naja_schematic` widget end to end in Colab. Run cells top to bottom; each **Check:** line says what to look for.

In [ ]:
!pip install -q naja-schematic
import najaeda, naja_schematic, anywidget
print("naja-schematic", naja_schematic.__version__, "| anywidget", anywidget.__version__)

## 1. Small hierarchical design

In [ ]:
%%writefile fulladder.v
module halfadder(input a, input b, output sum, output carry);
    and carry_and(carry, a, b);
    xor sum_xor(sum, a, b);
endmodule

module fulladder(input a, input b, input cin, output sum, output cout);
    wire sum1, carry1, carry2;
    halfadder ha1(.a(a), .b(b), .sum(sum1), .carry(carry1));
    halfadder ha2(.a(sum1), .b(cin), .sum(sum), .carry(carry2));
    or cout_or(cout, carry1, carry2);
endmodule

In [ ]:
from najaeda import netlist
netlist.reset()
top = netlist.load_verilog("fulladder.v")
top.get_name()

## 2. Basic view
**Check:** viewer renders; tree shows `fulladder`; expanding `ha1`/`ha2` lazily loads instances, terms and nets; right-click a term → *Show Equipotential* / *Trace to Driver* draws a schematic; clicking an open pin grows it; *Show Properties* fills the Properties tab.

In [ ]:
view = naja_schematic.show(height=600)
view

## 3. Diagnosis overlay (push into an existing view)
**Check:** in the view above, `ha1` turns red in the tree and schematic, the `sum1` wire/pin is tinted, and the Diagnosis panel lists both items.

In [ ]:
view.annotate([
    {"kind": "instance", "path": ["ha1"], "severity": "error",
     "message": "not equivalent", "source": "kepler-formal"},
    {"kind": "net", "path": ["ha1"], "terminal": "sum", "severity": "warning",
     "message": "suspicious output", "source": "colab-test"},
])

## 4. Host → viewer: focus an instance
**Check:** tree opens down to `ha2/sum_xor`, it's selected, Properties shows it, schematic draws it alone with open pins.

In [ ]:
view.show_instance("ha2/sum_xor")

In [ ]:
# Same thing with a najaeda Instance
view.show_instance(top.get_child_instance("ha1").get_child_instance("carry_and"))

## 5. Viewer → host: selection
Click an instance in the tree or a box in the schematic, then run the next cell. The callback prints on each click (output appears under this cell).

In [ ]:
handler = view.on_select(lambda inst: print("selected:", inst))

In [ ]:
print("selected_path:", view.selected_path)
inst = view.selected
print(inst, "->", inst.get_model_name() if inst else None)

## 6. One-shot view with diagnosis + instance

In [ ]:
naja_schematic.show(
    height=500,
    instance="ha1",
    diagnosis=[{"kind": "instance", "path": ["ha2"], "severity": "info",
                "message": "hello from show()", "source": "colab-test"}],
)

## 7. Edit, then a fresh view
**Check:** the new view shows `halfadder1` instead of `ha1` (the older view above does not refresh — known gap).

In [ ]:
top.get_child_instance("ha1").set_name("halfadder1")
naja_schematic.show(height=500)

## 8. Larger gate-level design with Liberty (tinyrocket)
**Check:** loads in reasonable time; tree scrolls; tracing a flop input to driver works and stays responsive.

In [ ]:
!wget -q https://github.com/najaeda/naja/archive/refs/heads/main.zip -O naja-main.zip && unzip -q -o naja-main.zip "naja-main/tutorials/benchmarks/*"
bench = "naja-main/tutorials/benchmarks"
netlist.reset()
netlist.load_liberty([f"{bench}/liberty/tutorial_cells.lib", f"{bench}/liberty/fakeram45_64x32.lib"])
top = netlist.load_verilog(f"{bench}/verilog/tinyrocket/tinyrocket.v")
naja_schematic.show(height=700)

## Things to note while testing
- Mouse wheel over the viewer doesn't scroll the page (known).
- Two/three views on the page at once: do they all stay interactive?
- Keyboard input only goes to the focused viewer?
- Any errors in the browser console (F12) or the Colab cell output?